In [2]:
import pandas as pd
import csv

df = pd.read_csv("data/amazon_reviews_us_Digital_Video_Games_v1_00.tsv.zip", sep="\t")
df.shape

(144724, 15)

Step 2: It loaded 144,724 rows and 15 columns.

In [3]:
df2 = pd.read_csv("data/amazon_reviews_us_Digital_Video_Games_v1_00.tsv.zip", sep="\t", quoting=csv.QUOTE_NONE)
print(df.shape)
print(df2.shape)
print(len(df2) - len(df))

(144724, 15)
(145431, 15)
707


Step 3: The first version loaded 144,724 rows and the second loaded 145,431, so 707 reviews were missing the first time. Pandas didn't give any warning about it. The stray quote marks made pandas think a bunch of rows were all one review.

This shows you can't just assume the data loaded right because there was no error. You should check the row count and look at the data before using it.

In [4]:
df = df2
five = df.sample(5, random_state=14)
for review in five["review_body"]:
    print(review[:600])
    print()

Google \\"SimCity 2013: The deception continues ...\\"<br /><br />- The presence of empty spaces not editable (useless empty spaces) with automatic road connections.<br />- Absence of terraforming.<br />- Absence of farms.<br />- Not being able to save your maps on the hard drive (you will not be able to reload the city to a stage prior to disasters).<br />- Road connections placed in a fixed and pre-determined location in the map.<br />- Airport City (the city is only one International Airport, occupying the entire map 2x2km).<br />- Absence of subway.<br />- Automatic road connections.<br />

Easy enough to jump into and can become difficulty as game continues. Though the side quests become tedious after a while, the game has decent replay ability. I still do not think I like it enough to purchase the (somewhat pricey) add-on (deepwater).

Finally, a port of the PS3 game to the PC.  Combines turn based strategy with shooting/action elements, set in an alternative &#34;Europe&#34; war

Step 4: There are html tags like `<br />` in the reviews, `&#34;` instead of quotation marks, and some quotes have extra backslashes in front of them. Some reviews are also really long and some are one line.

I think it came from the reviews being stored as html on Amazon's website and then exported into a text file, so the formatting stuff got left in.

In [5]:
df["star_rating"].value_counts().sort_index()

,count
star_rating,
1,24970
2,7749
3,11629
4,20406
5,80677


Step 5: Most of the ratings are 5 stars (about 55%), then 1 star (17%), 4 stars (14%), 3 stars (8%) and 2 stars (5%). So people mostly leave really good or really bad reviews.

This matters because a model could just guess positive every time and still be right around 70% of the time, so the accuracy would look better than the model actually is.

In [6]:
df["product_title"].value_counts().head(10)

,count
product_title,
Playstation Network Card,13642
Xbox Live Subscription,7307
Playstation Plus Subscription,4712
Xbox Live Gift Card,3438
SimCity - Limited Edition,3421
Xbox 360 Live Points Card,3036
Battlefield 4,1453
Final Fantasy XIV: A Realm Reborn,1273
Battlefield 3,1160


Step 6: A lot of the most reviewed "games" aren't actually games. They're PlayStation Network cards, Xbox Live subscriptions, PlayStation Plus and gift cards. The reviews for those are mostly about whether the code worked, not about a game.

In [7]:
print(df["review_body"].isna().sum())
print(df["review_body"].str.contains("<br />", regex=False).sum())

df = df.dropna(subset=["review_body"]).copy()
df["review_body"] = df["review_body"].str.replace("<br />", " ", regex=False)

print(df["review_body"].str.contains("<br />", regex=False).sum())

for review in df.loc[five.index, "review_body"]:
    print(review[:600])
    print()

3
25699
0
Google \\"SimCity 2013: The deception continues ...\\"  - The presence of empty spaces not editable (useless empty spaces) with automatic road connections. - Absence of terraforming. - Absence of farms. - Not being able to save your maps on the hard drive (you will not be able to reload the city to a stage prior to disasters). - Road connections placed in a fixed and pre-determined location in the map. - Airport City (the city is only one International Airport, occupying the entire map 2x2km). - Absence of subway. - Automatic road connections. - Player doesn't have complete control over the zo

Easy enough to jump into and can become difficulty as game continues. Though the side quests become tedious after a while, the game has decent replay ability. I still do not think I like it enough to purchase the (somewhat pricey) add-on (deepwater).

Finally, a port of the PS3 game to the PC.  Combines turn based strategy with shooting/action elements, set in an alternative &#34;Europ

Step 7: 25,699 reviews had line breaks before cleaning and 0 after. There were 3 empty reviews that got removed.

The `&#34;` and the backslashes are still there, and there are some double spaces where the line breaks were. I don't think it would change whether the model thinks a review is positive or negative since the sentiment comes from the actual words, not the symbols.

In [8]:
sample = df.groupby("star_rating").sample(100, random_state=42)
one_star = df[(df["star_rating"] == 1) & ~df.index.isin(sample.index)].sample(20, random_state=42)
sample["star_rating"].value_counts()

,count
star_rating,
1,100
2,100
3,100
4,100
5,100


Step 8: Since over half the reviews are 5 stars, 500 random reviews would be mostly 5 star reviews and only about 25 two star reviews. Taking 100 of each lets us see how the model does on every rating, not just the positive ones.

In [9]:
from transformers import pipeline

sentiment = pipeline("sentiment-analysis", model="distilbert/distilbert-base-uncased-finetuned-sst-2-english")

review = sample["review_body"].iloc[0]
print(sample["star_rating"].iloc[0], review)
sentiment(review)

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

1 Terrible, confusing interface. Looks like a cartoon. Didn't really like the long, drawn out stories in my player. Game play and graphics weren't that much improved $30 extra we had to pay (Now we get VC yea! not).


[{'label': 'NEGATIVE', 'score': 0.9998071789741516}]

Step 9: The keys are `label` and `score`. The label is POSITIVE or NEGATIVE and the score is how confident the model is in that label (0.9998 here).

The review was 1 star and the model said NEGATIVE, so it agrees with the customer.

In [10]:
%%time
results = sentiment(sample["review_body"].tolist(), truncation=True)
sample["label"] = [r["label"] for r in results]
sample["score"] = [r["score"] for r in results]

CPU times: user 1min 32s, sys: 120 ms, total: 1min 32s
Wall time: 1min 41s


Step 10: It took about 1 minute 41 seconds for 500 reviews, so about 0.2 seconds per review. At that rate all 145,000 would take around 8 hours.

truncation=True cuts off reviews that are longer than the model can handle (512 tokens). Without it the really long reviews would cause an error.

In [11]:
scored = sample[sample["star_rating"] != 3].copy()
scored["answer"] = "NEGATIVE"
scored.loc[scored["star_rating"] >= 4, "answer"] = "POSITIVE"
scored["correct"] = scored["label"] == scored["answer"]
scored["correct"].mean()

np.float64(0.8775)

Step 11: 3 star reviews are neutral so there isn't a right answer when the model can only say positive or negative.

The accuracy is about 87.8%. That's not bad, but I wouldn't trust it in a product by itself because about 1 in 8 reviews is wrong.

In [12]:
pd.crosstab(sample["star_rating"], sample["label"])

label,NEGATIVE,POSITIVE
star_rating,,
1,96,4
2,95,5
3,66,34
4,23,77
5,17,83


Step 12: 4 star reviews get wrong the most. 23 out of 100 were called negative. 5 stars were next with 17 wrong, while 1 and 2 stars were almost all right.

For the 3 star reviews it had to pick one, and it called 66 of them negative and 34 positive.

In [13]:
mistakes = scored[scored["correct"] == False]
for i, row in mistakes.sample(5, random_state=1).iterrows():
    print(row["star_rating"], row["label"], round(row["score"], 2))
    print(row["review_body"][:500])
    print()

4 NEGATIVE 0.92
A great game for any system. I had to create a steam account to play the game. Steam is free. I keep steam turned off cause it locks my computer. I also bought this game on the 3ds.

5 NEGATIVE 0.98
I bought this item and it didnt work ..till i read in the web that i must sign in with US playstation ID .. I inter my US adress with a new ID and it worked with me and i bought it 3 times  .. Thank you so much

5 NEGATIVE 0.96
What is there to say about this one? You get all three Bioshock games for $15!! I am a Steam fanboy so this was a no-brainer. You download the activation codes from Amazon, copy and paste them into Steam, download your games and play!

5 NEGATIVE 0.99
Fast and easy and more convenient then going to the store to buy the card. Got redemption code within seconds after purchasing it

1 POSITIVE 0.73
I wasted $7. im sure its a good game but i cant even get to the game part. i tried. a lot.



Step 13:
1. 4 stars, said negative. Says it's a great game but then complains about Steam. The model focused on the complaint. Model's fault.
2. 5 stars, said negative. Starts with "it didnt work" and ends with "thank you so much". The customer was happy so the model got it wrong.
3. 5 stars, said negative. Bioshock bundle for $15, "a no-brainer". This is clearly positive, the model probably got confused by the question at the start and the instructions part.
4. 5 stars, said negative. "Fast and easy... got redemption code within seconds". Clearly positive, the model doesn't understand that getting a code fast is a good thing.
5. 1 star, said positive (0.73). It says "im sure its a good game" but they couldn't get it to work. The wording is kind of misleading here but the model wasn't very confident.

Mostly the model's fault. The customers' ratings made sense.

In [14]:
%%time
stars = pipeline("sentiment-analysis", model="nlptown/bert-base-multilingual-uncased-sentiment")
results = stars(sample["review_body"].tolist(), truncation=True)
sample["pred_stars"] = [int(r["label"][0]) for r in results]

print((sample["pred_stars"] == sample["star_rating"]).mean())
print(((sample["pred_stars"] - sample["star_rating"]).abs() <= 1).mean())

config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  669MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/872k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

0.574
0.912
CPU times: user 3min 9s, sys: 2.16 s, total: 3min 11s
Wall time: 3min 16s


Step 14: It gets the exact star rating 57.4% of the time and is within one star 91.2% of the time.

I would use the star model for the dashboard. It gives a 1 to 5 score that matches the ratings the publisher already uses and it can handle neutral reviews. The first model looks more accurate but it can only say positive or negative and it called a lot of the 4 star reviews negative, which would make things look worse than they are.

In [15]:
%%time
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

labels = ["download or installation problems", "account or activation code issues", "bugs, crashes or performance", "boring or bad gameplay", "price or value"]
topics = classifier(one_star["review_body"].tolist(), candidate_labels=labels)
one_star["topic"] = [t["labels"][0] for t in topics]
one_star["topic"].value_counts()

config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

CPU times: user 3min 16s, sys: 4.2 s, total: 3min 20s
Wall time: 3min 42s


,count
topic,
download or installation problems,7
boring or bad gameplay,5
account or activation code issues,3
"bugs, crashes or performance",3
price or value,2


In [16]:
for t in topics[:3]:
    print(t["labels"][0], round(t["scores"][0], 2))
    print(t["sequence"][:400])
    print()

account or activation code issues 0.3
Didn't order it

bugs, crashes or performance 0.57
We never got this program to run properly!  Have returned it for a full refund without any problems.  The online game site is very tricky to use.  Would not use it again.

account or activation code issues 0.66
Never got the code, was still charged



Step 15: The most common complaint was download or installation problems (7), then boring or bad gameplay (5). Most of the complaints are about getting the game to work, not the game itself.

- "Didn't order it" got account or activation code issues (0.30). Doesn't really make sense but there's not much to go on, and the score is low.
- "We never got this program to run properly" got bugs, crashes or performance (0.57). Makes sense.
- "Never got the code, was still charged" got account or activation code issues (0.66). Makes sense.

In [17]:
labels2 = labels + ["technical problems", "cooking recipes"]
topics2 = classifier(one_star["review_body"].tolist()[:5], candidate_labels=labels2)

for a, b in zip(topics[:5], topics2):
    print(a["labels"][0], round(a["scores"][0], 2), "->", b["labels"][0], round(b["scores"][0], 2))

account or activation code issues 0.3 -> account or activation code issues 0.26
bugs, crashes or performance 0.57 -> technical problems 0.45
account or activation code issues 0.66 -> account or activation code issues 0.6
price or value 0.5 -> price or value 0.46
boring or bad gameplay 0.91 -> boring or bad gameplay 0.89


Step 16: I added "technical problems" (overlaps with bugs and download problems) and "cooking recipes" (irrelevant). The second review changed from bugs, crashes or performance (0.57) to technical problems (0.45). The other four kept the same top label but all the scores went down a little. Cooking recipes never won but still got a small score.

Since the scores have to add up to 1, adding more labels takes away from the other ones. Two labels that mean almost the same thing split the score between them, so the top label can change even though the review didn't.

For a real product you need to pick labels that don't overlap and actually match what people complain about, because the labels you pick change the results.